# Sensitivity Analysis Figures

This notebook generates Figures 1 (Calibration), 2 (Trajectories), and 4 (Algorithm comparison) for the three sensitivity analyses:
1. **tpt_60**: TPT completion reduced to 60% (vs 70% baseline)
2. **subclinical_50**: Subclinical TB prevalence set to 50% (vs calibrated baseline)
3. **homogeneous_mixing**: Homogeneous age-mixing assumption

Each sensitivity analysis has its own BCM object with independently calibrated parameters.

In [ ]:
# Import standard libraries
from pathlib import Path
from math import ceil
import numpy as np
import pandas as pd
import yaml

# Import scientific/plotting
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec

# Import project modules
from tbh.paths import REPO_ROOT_PATH
from tbh.model import get_tb_model
from tbh.plotting import plot_model_fit_with_uncertainty, plot_two_scenarios, plot_diff_outputs, title_lookup
import tbh.plotting as pl
import tbh.runner_tools as rt
from estival.model import BayesianCompartmentalModel

from importlib import reload

# Configure matplotlib
plt.style.use("seaborn-v0_8-white")
text_color = "#1f1f1f"
plt.rcParams.update({
    "font.family": "Helvetica",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "legend.frameon": False,
    "legend.fontsize": 7,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "text.color": text_color,
    "axes.labelcolor": text_color,
    "axes.titlecolor": text_color,
    "xtick.color": text_color,
    "ytick.color": text_color,
    "axes.edgecolor": text_color,
})

print("Imports successful")

In [ ]:
# Set paths
SA_BASE_DIR = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "59223189_sas"
OUTPUT_PARENT = REPO_ROOT_PATH / "notebooks" / "sa_figures"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)

# Scenario names for labeling (shared across all SAs)
SC_NAMES_CUSTOM = {
    'baseline': 'No screening',
    'scenario_1': 'PEARL / 65%',
    'scenario_2': 'PEARL / 75%',
    'scenario_3': 'PEARL algorithm at 85% coverage',
    'scenario_6': 'CXR-TST / 65%',
    'scenario_7': 'CXR-TST / 75%',
    'scenario_8': 'CXR-TST / 85%',
    'scenario_16': 'CXR / 65%',
    'scenario_17': 'CXR / 75%',
    'scenario_18': 'CXR / 85%'
}

# Figure-specific outputs for Figure 2 (trajectories)
TRAJECTORY_OUTPUTS = [
    "tb_incidence_per100k",
    "viable_tbi_prevalence_perc",
    "tb_prevalence_per100k",
    "tb_mortality_per100k",
]

# Scenarios for Figure 4 (algorithm comparison)
SCENARIOS_FIG4 = ["scenario_1", "scenario_2", "scenario_3", "scenario_6", "scenario_7", "scenario_8", "scenario_16", "scenario_17", "scenario_18"]

# Map task number to SA name and description
SA_MAP = {
    1: {"name": "tpt_60", "desc": "TPT completion 60%"},
    2: {"name": "subclinical_50", "desc": "Subclinical prevalence 50%"},
    3: {"name": "homogeneous_mixing", "desc": "Homogeneous mixing"},
}

print(f"SA base directory: {SA_BASE_DIR}")
print(f"Output directory: {OUTPUT_PARENT}")

In [ ]:
# Load the SA configuration map
config_map_path = SA_BASE_DIR / "sa_config_map.yaml"
with open(config_map_path, "r") as f:
    sa_config = yaml.safe_load(f)

print(f"SA configuration loaded: {sa_config}")

# Verify task directories exist
task_paths = {}
for task_id in SA_MAP.keys():
    task_dir = SA_BASE_DIR / f"task_{task_id}"
    if task_dir.exists():
        task_paths[task_id] = task_dir
        print(f"Task {task_id} ({SA_MAP[task_id]['name']}): {task_dir}")
    else:
        print(f"WARNING: Task {task_id} directory not found: {task_dir}")

print(f"\nFound {len(task_paths)} tasks")

In [ ]:
def make_figure_1_calibration_no_pdf(uncertainty_df, bcm, colour="#B22222", figsize=(10.8, 5.5)):
    """Generate Figure 1: Calibration panel with uncertainty (no PDF image as top panel)."""
    selected_outputs = [
        "pearl_posXreach_reachable_per100k",
        "cxr_posXreach_reachable_per100k",
        "perc_prev_subclinicalXreach_reachable",
        "perc_prev_infectiousXreach_reachable",
        "notifications",
    ]
    
    n_col = 3
    n_data_panels = len(selected_outputs) + 1  # 6 data panels total (5 outputs + 1 TST age-stratified)
    n_data_row = ceil(n_data_panels / n_col)  # 2 rows for data
    
    # Create gridspec without the PDF row
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(n_data_row, n_col, figure=fig,
                           hspace=.2, wspace=0.3)
    
    # Create axes for data panels
    axes = []
    for i in range(n_data_panels):
        row = i // n_col
        col = i % n_col
        ax = fig.add_subplot(gs[row, col])
        axes.append(ax)
    
    # Plot the selected outputs
    for i, output in enumerate(selected_outputs):
        ax = axes[i]
        x_min = 1995 if output == "notifications" else 2010
        pl.plot_model_fit_with_uncertainty(ax, uncertainty_df, output, bcm, x_lim=(x_min, 2025), colour=colour, target_ms=15)
        if i == 0:
            ax.legend()
    
    # Age-stratified TST positivity panel
    ax = axes[len(selected_outputs)]
    agegroups = ["3_9", "10", "15+", "18+"]
    model_median, model_low, model_high, observed, x_tick_labels = [], [], [], [], []
    
    for age in agegroups:
        output_name = f"tst_posXage_{age}Xreach_reachable_perc"
        year = bcm.targets[output_name].data.index[0]
        q = uncertainty_df[output_name].loc[year]
        obs = bcm.targets[output_name].data.iloc[0]
        
        model_median.append(q["0.5"])
        model_low.append(q["0.025"])
        model_high.append(q["0.975"])
        observed.append(obs)
        
        suffix = f" y.o.\n({year})"
        if age == "3_9":
            x_tick_labels.append("3-9" + suffix)
        elif age == "15+":
            x_tick_labels.append("15+" + suffix)
        else:
            x_tick_labels.append(f"{age}" + suffix)
    
    x = range(len(agegroups))
    ax.errorbar(
        [i - 0.06 for i in x],
        model_median,
        yerr=[
            [m - l for m, l in zip(model_median, model_low)],
            [h - m for h, m in zip(model_high, model_median)],
        ],
        fmt="D",
        color=colour,
        ecolor=colour,
        markersize=3,
        elinewidth=2.0,
        capsize=0,
        label="Model (median, 95% CrI)",
    )
    ax.scatter([i + 0.06 for i in x], observed, color="black", s=7, zorder=5, label="Observed")
    ax.set_xticks(list(x))
    ax.set_xticklabels(x_tick_labels)
    ax.set_ylabel(title_lookup["tst_posXreach_reachable_perc"])
    
    model_handle = mlines.Line2D([], [], color=colour, marker="D", markersize=3, linestyle="-", label="Model (median, 95% CrI)")
    obs_handle = mlines.Line2D([], [], color="black", marker="o", linestyle="None", markersize=3, label="Observed")
    ax.legend(handles=[obs_handle, model_handle], frameon=False, loc="best")
    
    # Panel letters
    letter_fontsize = 9
    for i, ax in enumerate(axes):
        ax.text(
            -0.15,
            1.1,
            f"{chr(97 + i)})",
            transform=ax.transAxes,
            fontsize=letter_fontsize,
            fontweight="bold",
            va="top",
        )
    
    return fig

print("Figure 1 function defined")

In [ ]:
def make_figure_4_algorithms(diff_dfs, figsize=(5, 5)):
    """Generate Figure 4: Algorithm and coverage comparison."""
    reload(pl)
    
    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=False)
    
    group_labels = ["PEARL (CXR-Xpert-TST)", "CXR-TST", "Disease screening only (CXR)"]
    coverage_labels = ["65%", "75%", "85%"]
    xtick_labels = coverage_labels * len(group_labels)
    
    def format_fig4_axis(ax):
        ax.set_xticks(range(1, len(SCENARIOS_FIG4) + 1), xtick_labels)
        plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
        
        for boundary in (3.5, 6.5):
            ax.axvline(boundary, color="0.5", linestyle="--", linewidth=0.9, alpha=0.9, zorder=0)
        
        group_centers = [2, 5, 8]
        for center, label in zip(group_centers, group_labels):
            ax.text(
                center,
                .98,
                label,
                ha="center",
                va="bottom",
                transform=ax.get_xaxis_transform(),
                fontsize=7,
            )
        ax.set_xlabel("Screening coverage")
        ax.set_ylim(0, 65)
    
    # Panel A: TB episodes averted
    ax = axes[0]
    pl.plot_diff_outputs(ax, diff_dfs, "TB_averted_relative", SCENARIOS_FIG4, colour="#B22222")
    format_fig4_axis(ax)
    ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
    ax.text(-0.06, 1.08, "a)", transform=ax.transAxes, fontsize=9, fontweight='bold', va='top')
    
    # Panel B: TB deaths averted
    ax = axes[1]
    pl.plot_diff_outputs(ax, diff_dfs, "deaths_averted_relative", SCENARIOS_FIG4, colour="#B22222")
    format_fig4_axis(ax)
    ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
    ax.text(-0.06, 1.08, "b)", transform=ax.transAxes, fontsize=9, fontweight='bold', va='top')
    
    fig.tight_layout()
    return fig

print("Figure 4 function defined")

In [ ]:
# Loop through each sensitivity analysis task and generate figures
for task_id, task_path in task_paths.items():
    sa_name = SA_MAP[task_id]['name']
    sa_desc = SA_MAP[task_id]['desc']
    
    print(f"\n{'='*70}")
    print(f"Processing Task {task_id}: {sa_name} ({sa_desc})")
    print(f"{'='*70}")
    
    # Create output subdirectory for this SA
    sa_output_dir = OUTPUT_PARENT / sa_name
    sa_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {sa_output_dir}")
    
    # Load uncertainty dataframes
    try:
        unc_baseline = pd.read_parquet(task_path / "uncertainty_df_baseline.parquet")
        unc_scenario3 = pd.read_parquet(task_path / "uncertainty_df_scenario_3.parquet")
        print("✓ Uncertainty data loaded")
    except FileNotFoundError as e:
        print(f"✗ Failed to load uncertainty data: {e}")
        continue
    
    # Load BCM for calibration figure
    try:
        params, priors, tv_params = rt.get_parameters_and_priors()
        
        with open(task_path / "details.yaml", "r") as f:
            docs = list(yaml.safe_load_all(f))
        
        model_config = docs[1] if len(docs) > 1 else {}
        if not isinstance(model_config, dict):
            model_config = {}
        
        model = get_tb_model(model_config, tv_params)
        
        # For homogeneous_mixing, remove the mixing_matrix_distance target
        calibration_targets = rt.targets.copy()
        if sa_name == "homogeneous_mixing":
            calibration_targets = [t for t in calibration_targets if t.name != "mixing_matrix_distance"]
        
        bcm = BayesianCompartmentalModel(model, params, priors, calibration_targets)
        print("✓ BCM loaded for calibration")
    except Exception as e:
        print(f"✗ Failed to load BCM: {e}")
        continue
    
    # ========== FIGURE 1: Calibration ==========
    try:
        print("\nGenerating Figure 1 (Calibration)...")
        fig1 = make_figure_1_calibration_no_pdf(unc_baseline, bcm)
        plt.savefig(sa_output_dir / "figure_1_calibration.png", dpi=300, bbox_inches='tight')
        plt.savefig(sa_output_dir / "figure_1_calibration.pdf", bbox_inches='tight')
        plt.close(fig1)
        print("✓ Figure 1 saved")
    except Exception as e:
        print(f"✗ Figure 1 failed: {e}")
    
    # ========== FIGURE 2: Trajectories ==========
    try:
        print("Generating Figure 2 (Trajectories)...")
        fig2 = make_figure_2_trajectories(unc_baseline, unc_scenario3)
        plt.savefig(sa_output_dir / "figure_2_trajectories.png", dpi=300, bbox_inches='tight')
        plt.savefig(sa_output_dir / "figure_2_trajectories.pdf", bbox_inches='tight')
        plt.close(fig2)
        print("✓ Figure 2 saved")
    except Exception as e:
        print(f"✗ Figure 2 failed: {e}")
    
    # ========== FIGURE 4: Algorithm Comparison ==========
    try:
        print("Generating Figure 4 (Algorithm comparison)...")
        
        # Load diff outputs for algorithm comparison
        diff_dfs = {}
        for scenario in SCENARIOS_FIG4:
            diff_path = task_path / f"diff_quantiles_df_ref_baseline_{scenario}.parquet"
            if diff_path.exists():
                diff_dfs[scenario] = pd.read_parquet(diff_path)
        
        if len(diff_dfs) == len(SCENARIOS_FIG4):
            fig4 = make_figure_4_algorithms(diff_dfs)
            plt.savefig(sa_output_dir / "figure_4_algorithms_coverage.png", dpi=300, bbox_inches='tight')
            plt.savefig(sa_output_dir / "figure_4_algorithms_coverage.pdf", bbox_inches='tight')
            plt.close(fig4)
            print(f"✓ Figure 4 saved (loaded {len(diff_dfs)} scenario diffs)")
        else:
            print(f"✗ Figure 4 failed: only found {len(diff_dfs)}/{len(SCENARIOS_FIG4)} scenario diff files")
    except Exception as e:
        print(f"✗ Figure 4 failed: {e}")

print(f"\n{'='*70}")
print("All sensitivity analysis figures completed!")
print(f"Outputs saved to: {OUTPUT_PARENT}")
print(f"{'='*70}")

In [ ]:
# Generate parameter posterior vs prior comparison figures for each SA
print(f"\n{'='*70}")
print("Generating parameter posterior figures...")
print(f"{'='*70}")

for task_id, task_path in task_paths.items():
    sa_name = SA_MAP[task_id]['name']
    sa_desc = SA_MAP[task_id]['desc']
    
    print(f"\nProcessing posteriors for {sa_name}...")
    
    sa_output_dir = OUTPUT_PARENT / sa_name
    
    try:
        # Load idata
        idata = az.from_netcdf(task_path / "idata.nc")
        
        # Determine burn-in from details.yaml
        with open(task_path / "details.yaml", "r") as f:
            docs = list(yaml.safe_load_all(f))
        
        analysis_config = docs[2] if len(docs) > 2 else {}
        if not isinstance(analysis_config, dict):
            analysis_config = {}
        
        burn_in = analysis_config.get('burn_in', 5000)
        
        # Reload priors for this SA
        params, priors, tv_params = rt.get_parameters_and_priors()
        
        # For homogeneous_mixing, remove the mixing priors
        if sa_name == "homogeneous_mixing":
            priors = [p for p in priors if p.name not in ['bg_mixing', 'a_spread', 'pc_strength']]
        
        # Filter priors to only those present in idata
        posterior_vars = set(idata.posterior.data_vars.keys())
        prior_names = [p.name for p in priors if p.name in posterior_vars]
        prior_objs = [p for p in priors if p.name in posterior_vars]
        
        # Generate posterior vs prior comparison figure
        fig = pl.plot_post_prior_comparison(
            idata, 
            burn_in, 
            prior_names, 
            prior_objs, 
            n_col=4
        )
        
        plt.savefig(sa_output_dir / "parameter_posteriors.png", dpi=300, bbox_inches='tight')
        plt.savefig(sa_output_dir / "parameter_posteriors.pdf", bbox_inches='tight')
        plt.close(fig)
        print(f"✓ Parameter posteriors saved for {sa_name}")
        
    except Exception as e:
        print(f"✗ Failed to generate posteriors for {sa_name}: {e}")

print(f"\n{'='*70}")
print("Parameter posterior generation complete!")
print(f"{'='*70}")

## Parameter Posteriors

## Generate Figures for Each Sensitivity Analysis

In [ ]:
def make_figure_2_trajectories(unc_baseline, unc_scenario3, figsize=(7, 4.65)):
    """Generate Figure 2: Projected trajectories (baseline vs scenario_3)."""
    reload(pl)
    
    unc_dfs = {
        "baseline": unc_baseline,
        "scenario_3": unc_scenario3,
    }
    
    unc_sc_colours = ["#B22222", "#54992c"]
    
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=False)
    axes = axes.flatten()
    
    for ax, output in zip(axes, TRAJECTORY_OUTPUTS):
        pl.plot_two_scenarios(
            ax,
            unc_dfs,
            output,
            scenarios=["baseline", "scenario_3"],
            xlim=(2020, 2035),
            include_unc=True,
            ylab_fontsize=9,
            unc_sc_colours=unc_sc_colours,
            include_legend=ax==axes[0],
            sc_names=SC_NAMES_CUSTOM
        )
    
    # Panel letters
    panel_letters = [f"{chr(97 + i)})" for i in range(len(TRAJECTORY_OUTPUTS))]
    for i, letter in enumerate(panel_letters):
        axes[i].text(
            -0.15,
            1.05,
            letter,
            transform=axes[i].transAxes,
            fontsize=9,
            fontweight="bold",
            va="top",
        )
    
    fig.tight_layout()
    return fig

print("Figure 2 function defined")

## Define Figure Generation Functions

## Load Sensitivity Analysis Configuration

## Configuration